----
<div style="display: flex; align-items: center;">
    <img src="https://developers.lseg.com/content/dam/devportal/icons/logo/lseg-logo.svg" width="20%" style="vertical-align: top;">
    <h1 style="margin-left: 20px;">Data Library for Python</h1>
</div>

----

## Content layer - Historical Pricing with real-time updates

Use a single `historical_pricing.summaries.Definition().get_stream()` to receive the initial history and subsequent real-time summaries. The first `on_insert` notification initializes one curve per requested field; subsequent notifications update those same curves. This example extends [Historical Pricing](EX-2.01.01-HistoricalPricing.ipynb) using the event-driven approach introduced in [Pricing - Streaming events](../2.02-Pricing/EX-2.02.04-Pricing-StreamingEvents.ipynb).

Run this notebook in Jupyter with a version of `lseg-data` supporting Historical Pricing summary streams, `pandas` and `matplotlib` installed, and a configured session entitled to Historical Pricing and real-time data. The plot uses an updatable notebook display; no widget extension is required.

## Set the configuration file location

In [1]:
import os
os.environ["LD_LIB_CONFIG_PATH"] = "../../../Configuration"

## Imports

In [2]:
import asyncio
from datetime import timezone

import lseg.data as ld
from lseg.data.content import historical_pricing
from lseg.data.content.historical_pricing import Intervals
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

## Open the data session

Edit `lseg-data.config.json` to select your session and connection parameters.

In [3]:
ld.open_session()

C:\Users\pfaurel\AppData\Roaming\Python\Python312\site-packages\lseg\data\_configure.py:524:UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


<lseg.data.session.Definition object at 0x1943f7ff380 {name='workspace'}>

## Configure the Historical Pricing stream

Select one instrument and numeric fields. The same stream delivers the initial 120 one-minute summaries and subsequent real-time summary notifications. There is no separate `get_data()` request or Pricing stream.

In [4]:
instrument = ["EUR="]
fields = ["BID", "ASK"]

stream = historical_pricing.summaries.Definition(
    universe=instrument,
    fields=fields,
    interval=Intervals.ONE_MINUTE,
    count=120,
).get_stream()

## Define the plotter

The first Insert creates each Matplotlib line from its historical series. Subsequent Inserts merge new rows by timestamp, and Updates replace only the published fields of an existing bar. Each notification redraws **every** curve without recreating the lines. Fields absent from a partial notification keep their existing values; explicit nulls produce gaps. An overlapping Insert replaces the supplied values rather than creating duplicate timestamps.

The plot keeps all received points and redraws on every event for clarity. This is a short-running demonstration, not a throttled or bounded-memory dashboard.

In [5]:
class HistoricalRealtimePlotter:
    def __init__(self, instrument, fields, history):
        self.history = history.loc[:, fields].copy()
        if self.history.empty:
            raise ValueError(f"No historical data returned for {instrument}.")
        if self.history.isna().all().any():
            missing_fields = self.history.columns[self.history.isna().all()].tolist()
            raise ValueError(f"No historical values returned for fields: {missing_fields}")

        with plt.ioff():
            self.figure, self.axes = plt.subplots(figsize=(12, 5))
        self.lines = {
            field: self.axes.plot(
                self.history.index.to_pydatetime(),
                self.history[field].to_numpy(dtype=float, na_value=float("nan")),
                label=field,
            )[0]
            for field in fields
        }
        self.axes.set_title(f"{instrument} - Historical and real-time pricing")
        self.axes.set_xlabel("Time (UTC)")
        self.axes.set_ylabel("Value")
        self.axes.xaxis.set_major_formatter(
            mdates.DateFormatter("%H:%M:%S", tz=timezone.utc)
        )
        self.axes.legend()
        self.axes.grid(True, alpha=0.3)
        self.figure.autofmt_xdate()
        self.figure.tight_layout()
        self.display_handle = display(self.figure, display_id=True)
        plt.close(self.figure)

    def update(self, frame):
        self.history = self.history.reindex(self.history.index.union(frame.index).sort_values())
        self.history.loc[frame.index, frame.columns] = frame
        for field, line in self.lines.items():
            line.set_data(
                self.history.index.to_pydatetime(),
                self.history[field].to_numpy(dtype=float, na_value=float("nan")),
            )

        self.axes.relim()
        self.axes.autoscale_view()
        self.display_handle.update(self.figure)

## Define the streaming event handlers

`on_insert` receives `(response, stream)`, with the initial history or newly inserted summaries in `response.data.df`. `on_update` receives `(data, stream)`, with changed fields of the current summary in `data.df`. Both feed the same plotter. Timestamps come from the summary index and are normalized to UTC, not replaced with local reception times.

Callbacks copy the received DataFrame and schedule plotting on the notebook's asyncio loop with `call_soon_threadsafe`, so Matplotlib is never modified by the streaming thread. Status and error notifications remain visible in the notebook.

In [6]:
notebook_loop = asyncio.get_running_loop()
plotter = None
plotting_active = True


def plot_summaries(frame):
    global plotter
    if not plotting_active:
        return

    frame = frame.loc[:, frame.columns.intersection(fields)].copy()
    if frame.empty:
        raise ValueError("The summary notification contains no requested data.")
    frame.index = pd.to_datetime(frame.index, utc=True)
    frame = frame.loc[~frame.index.duplicated(keep="last")].sort_index()
    frame = frame.apply(pd.to_numeric, errors="raise")
    if plotter is None:
        plotter = HistoricalRealtimePlotter(instrument, fields, frame)
    else:
        plotter.update(frame)


def plot_inserted_summaries(response, historical_stream):
    notebook_loop.call_soon_threadsafe(plot_summaries, response.data.df.copy(deep=True))


def plot_updated_summary(data, historical_stream):
    notebook_loop.call_soon_threadsafe(plot_summaries, data.df.copy(deep=True))


def display_status(status, historical_stream):
    notebook_loop.call_soon_threadsafe(print, "Historical Pricing status:", status)


def display_error(error, historical_stream):
    notebook_loop.call_soon_threadsafe(print, "Historical Pricing error:", error)

## Register callbacks and open the Historical Pricing stream

Register the callbacks before opening the stream so the initial historical Insert is captured. Run this cell and keep the notebook open to see the chart appear and change as summaries arrive. Do not immediately run the cleanup cell, otherwise streaming stops. Close the existing stream before rerunning the setup cells.

In [24]:
stream.on_insert(plot_inserted_summaries)
stream.on_update(plot_updated_summary)
stream.on_status(display_status)
stream.on_error(display_error)
plotting_active = True
stream.open()

<OpenState.Opened: 'Opened'>

## Stop streaming and close the session

Run this cell manually when finished. Disabling plotting also ignores notifications already scheduled on the notebook loop, even if the initial history has not arrived yet. The last displayed chart remains visible.

In [25]:
plotting_active = False
stream.close()  

<OpenState.Closed: 'Closed'>

In [ ]:
ld.close_session()